# STEP 1: LOAD CLEAN FILES

In [14]:
import pandas as pd

# =========================
# STEP 1: LOAD CLEAN FILES
# =========================
movies_ref = pd.read_csv(r"C:\Users\Computop\Downloads\movies_reference_clean.csv")
tmdb = pd.read_csv(r"C:\Users\Computop\Downloads\tmdb_clean.csv")
boxoffice = pd.read_csv(r"C:\Users\Computop\Downloads\boxoffice_clean.csv")

print("Files loaded successfully.\n")

print("Movies Reference columns:")
print(movies_ref.columns.tolist(), "\n")

print("TMDB columns:")
print(tmdb.columns.tolist(), "\n")

print("Box Office columns:")
print(boxoffice.columns.tolist(), "\n")

Files loaded successfully.

Movies Reference columns:
['Rank', 'Title', 'Theaters', 'Total Gross', 'Release Date', 'Distributor', 'Year', 'title_clean', 'is_rerelease'] 

TMDB columns:
['genre_ids', 'id', 'original_language', 'original_title', 'popularity', 'release_date', 'title', 'vote_average', 'vote_count', 'year', 'genres', 'title_clean'] 

Box Office columns:
['rank', 'title', 'revenue', 'year', 'title_clean'] 



# STEP 2: FIX COLUMN NAMES


In [15]:
# =========================
# STEP 2: FIX COLUMN NAMES
# =========================
movies_ref = movies_ref.rename(columns={
    "Year": "year",
    "Title": "title_ref"
})

print("Updated Movies Reference columns:")
print(movies_ref.columns.tolist(), "\n")

Updated Movies Reference columns:
['Rank', 'title_ref', 'Theaters', 'Total Gross', 'Release Date', 'Distributor', 'year', 'title_clean', 'is_rerelease'] 



# STEP 3: CHECK KEY COLUMNS


In [16]:
# =========================
# STEP 3: CHECK REQUIRED KEYS
# =========================
required_cols = ["title_clean", "year"]

for name, df in [("movies_ref", movies_ref), ("tmdb", tmdb), ("boxoffice", boxoffice)]:
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{col} is missing in {name}")

print("All required key columns exist: title_clean, year\n")

All required key columns exist: title_clean, year



# STEP 4: STANDARDIZE TYPES


In [17]:
# =========================
# STEP 4: STANDARDIZE YEAR
# =========================
movies_ref["year"] = pd.to_numeric(movies_ref["year"], errors="coerce")
tmdb["year"] = pd.to_numeric(tmdb["year"], errors="coerce")
boxoffice["year"] = pd.to_numeric(boxoffice["year"], errors="coerce")

movies_ref = movies_ref.dropna(subset=["title_clean", "year"])
tmdb = tmdb.dropna(subset=["title_clean", "year"])
boxoffice = boxoffice.dropna(subset=["title_clean", "year"])

movies_ref["year"] = movies_ref["year"].astype(int)
tmdb["year"] = tmdb["year"].astype(int)
boxoffice["year"] = boxoffice["year"].astype(int)

print("Year column standardized.\n")

Year column standardized.



# STEP 5: MERGE DATASETS

In [18]:
# =========================
# STEP 5: MAIN MERGE
# =========================
main_df = pd.merge(
    tmdb,
    boxoffice,
    on=["title_clean", "year"],
    how="inner",
    suffixes=("_tmdb", "_box")
)

print("Matched TMDB + Box Office:", len(main_df), "\n")

print("Main merged columns:")
print(main_df.columns.tolist(), "\n")

print("Preview of main merged dataset:")
print(main_df.head())

Matched TMDB + Box Office: 281 

Main merged columns:
['genre_ids', 'id', 'original_language', 'original_title', 'popularity', 'release_date', 'title_tmdb', 'vote_average', 'vote_count', 'year', 'genres', 'title_clean', 'rank', 'title_box', 'revenue'] 

Preview of main merged dataset:
                 genre_ids      id original_language  \
0  [10751, 35, 12, 14, 16]  502356                en   
1                     [18]     615                en   
2                 [18, 35]     350                en   
3            [28, 12, 878]  634649                en   
4            [12, 18, 878]  157336                en   

                original_title  popularity release_date  \
0  The Super Mario Bros. Movie    380.5992   2023-04-05   
1    The Passion of the Christ    130.3440   2004-02-25   
2        The Devil Wears Prada     99.2223   2006-06-29   
3      Spider-Man: No Way Home     77.2189   2021-12-15   
4                 Interstellar     67.6472   2014-11-05   

                    ti

# STEP 6: TRACK MATCHED / UNMATCHED


In [19]:
# =========================
# STEP 6: ADD REFERENCE CSV AS SUPPLEMENTARY
# =========================
final_df = pd.merge(
    main_df,
    movies_ref,
    on=["title_clean", "year"],
    how="left"
)

print("Final dataset rows after adding reference data:", len(final_df), "\n")

print("Final merged columns:")
print(final_df.columns.tolist(), "\n")

print("Preview of final merged dataset:")
print(final_df.head())

Final dataset rows after adding reference data: 281 

Final merged columns:
['genre_ids', 'id', 'original_language', 'original_title', 'popularity', 'release_date', 'title_tmdb', 'vote_average', 'vote_count', 'year', 'genres', 'title_clean', 'rank', 'title_box', 'revenue', 'Rank', 'title_ref', 'Theaters', 'Total Gross', 'Release Date', 'Distributor', 'is_rerelease'] 

Preview of final merged dataset:
                 genre_ids      id original_language  \
0  [10751, 35, 12, 14, 16]  502356                en   
1                     [18]     615                en   
2                 [18, 35]     350                en   
3            [28, 12, 878]  634649                en   
4            [12, 18, 878]  157336                en   

                original_title  popularity release_date  \
0  The Super Mario Bros. Movie    380.5992   2023-04-05   
1    The Passion of the Christ    130.3440   2004-02-25   
2        The Devil Wears Prada     99.2223   2006-06-29   
3      Spider-Man: No W

# STEP 7: CHECK FINAL COLUMNS


In [20]:
# =========================
# STEP 7: TRACK MATCHED / UNMATCHED
# =========================
main_matched_keys = set(zip(main_df["title_clean"], main_df["year"]))

tmdb_keys = set(zip(tmdb["title_clean"], tmdb["year"]))
boxoffice_keys = set(zip(boxoffice["title_clean"], boxoffice["year"]))
movies_ref_keys = set(zip(movies_ref["title_clean"], movies_ref["year"]))

print("Matched TMDB + Box Office:", len(main_matched_keys))
print("Unmatched TMDB:", len(tmdb_keys - main_matched_keys))
print("Unmatched Box Office:", len(boxoffice_keys - main_matched_keys))

# Overlap with reference file
ref_overlap = movies_ref[
    movies_ref.apply(lambda r: (r["title_clean"], r["year"]) in main_matched_keys, axis=1)
]

print("Reference CSV overlap with main dataset:", len(ref_overlap))
print("Reference CSV unmatched:", len(movies_ref) - len(ref_overlap), "\n")

Matched TMDB + Box Office: 281
Unmatched TMDB: 219
Unmatched Box Office: 719
Reference CSV overlap with main dataset: 13
Reference CSV unmatched: 187 



In [21]:
# =========================
# STEP 8: SAVE UNMATCHED FILES
# =========================
unmatched_tmdb = tmdb[
    ~tmdb.apply(lambda r: (r["title_clean"], r["year"]) in main_matched_keys, axis=1)
]

unmatched_boxoffice = boxoffice[
    ~boxoffice.apply(lambda r: (r["title_clean"], r["year"]) in main_matched_keys, axis=1)
]

unmatched_movies_ref = movies_ref[
    ~movies_ref.apply(lambda r: (r["title_clean"], r["year"]) in main_matched_keys, axis=1)
]

unmatched_tmdb.to_csv(r"C:\Users\Computop\Downloads\unmatched_tmdb.csv", index=False)
unmatched_boxoffice.to_csv(r"C:\Users\Computop\Downloads\unmatched_boxoffice.csv", index=False)
unmatched_movies_ref.to_csv(r"C:\Users\Computop\Downloads\unmatched_movies_reference.csv", index=False)

print("Unmatched files saved successfully.\n")

Unmatched files saved successfully.



# STEP 8: CREATE FINAL DATASET


In [22]:
# =========================
# STEP 9: CREATE FINAL DATASET
# =========================
final_dataset = pd.DataFrame()

# Main title column
if "title_box" in final_df.columns:
    final_dataset["title"] = final_df["title_box"]
elif "title_tmdb" in final_df.columns:
    final_dataset["title"] = final_df["title_tmdb"]
elif "title_ref" in final_df.columns:
    final_dataset["title"] = final_df["title_ref"]

# Core fields
final_dataset["title_clean"] = final_df["title_clean"]
final_dataset["year"] = final_df["year"]

# Main metrics
if "revenue" in final_df.columns:
    final_dataset["revenue"] = final_df["revenue"]

if "vote_average" in final_df.columns:
    final_dataset["vote_average"] = final_df["vote_average"]

if "popularity" in final_df.columns:
    final_dataset["popularity"] = final_df["popularity"]

if "genres" in final_df.columns:
    final_dataset["genres"] = final_df["genres"]

# Supplementary reference fields
if "Rank" in final_df.columns:
    final_dataset["rank_ref"] = final_df["Rank"]

if "Distributor" in final_df.columns:
    final_dataset["distributor"] = final_df["Distributor"]

if "Total Gross" in final_df.columns:
    final_dataset["total_gross_ref"] = final_df["Total Gross"]

if "Theaters" in final_df.columns:
    final_dataset["theaters_ref"] = final_df["Theaters"]

if "Release Date" in final_df.columns:
    final_dataset["release_date_ref"] = final_df["Release Date"]

final_dataset.to_csv(r"C:\Users\Computop\Downloads\final_dataset.csv", index=False)

print("final_dataset.csv created successfully.")
print("Final dataset shape:", final_dataset.shape)
print("\nPreview:")
print(final_dataset.head())

final_dataset.csv created successfully.
Final dataset shape: (281, 12)

Preview:
                         title                  title_clean  year    revenue  \
0  The Super Mario Bros. Movie  the super mario bros. movie  2023  574934330   
1    The Passion of the Christ    the passion of the christ  2004  370782930   
2        The Devil Wears Prada        the devil wears prada  2006  124740460   
3      Spider-Man: No Way Home      spider-man: no way home  2021  814866759   
4                 Interstellar                 interstellar  2014  203227580   

   vote_average  popularity                                         genres  \
0         7.588    380.5992  Family, Comedy, Adventure, Fantasy, Animation   
1         7.537    130.3440                                          Drama   
2         7.389     99.2223                                  Drama, Comedy   
3         7.934     77.2189             Action, Adventure, Science Fiction   
4         8.500     67.6472              Adventu

A strict 3-way join across TMDB, Box Office Mojo, and the reference CSV produced only 13 matched records, due to differences in dataset coverage and overlap. To preserve data quality and analytical usefulness, TMDB and Box Office Mojo were used to create the primary integrated dataset, producing 281 matched records. The reference CSV was retained as a supplementary source and joined optionally for enrichment where overlap existed.